In [4]:
# ─── 1. CÀI ĐẶT MÔI TRƯỜNG (ĐÃ FIX LỖI NUMPY COPY=FALSE) ─────────────────────
!pip uninstall -y peft accelerate transformers datasets numpy pyarrow -q
!pip install -q \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.20.0 \
  safetensors==0.4.2 \
  scikit-learn \
  Pillow \
  "numpy<2.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but yo

In [1]:
# ─── 2. SETUP & IMPORTS ───────────────────────────────────────────────────────
import os, time, functools
import torch
import numpy as np
from PIL import Image
from torch import nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import (CLIPProcessor, CLIPModel, TrainingArguments,
                          Trainer, EarlyStoppingCallback)
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers.trainer_utils import get_last_checkpoint
from datasets import load_dataset
from google.colab import drive

# Tối ưu bộ nhớ GPU cực hạn cho mô hình Large
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Kết nối Drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/CLIP_ViT_L14_IFND_Final"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Save to:", SAVE_DIR)

# ─── 3. DATASET & PREPROCESS ──────────────────────────────────────────────────
print("⏳ Loading dataset IFND-multimodal...")
hf_dataset = load_dataset("Nhat243/IFND-multimodal")

model_name = "openai/clip-vit-large-patch14" # Bản Large L/14
processor  = CLIPProcessor.from_pretrained(model_name)

def preprocess(example):
    try:
        image = example['image'].convert("RGB")
    except:
        image = Image.new("RGB", (224, 224), (128, 128, 128))

    inputs = processor(
        text=str(example['text']),
        images=image,
        truncation=True,
        padding="max_length",
        max_length=77,
        return_tensors="pt"
    )
    return {
        "input_ids":      inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "pixel_values":   inputs["pixel_values"][0],
        "labels":         int(example["label"])
    }

print("⏳ Preprocessing dataset...")
encoded_dataset = hf_dataset.map(
    preprocess,
    remove_columns=hf_dataset["train"].column_names,
    desc="Preprocessing"
)
encoded_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "pixel_values", "labels"])

train_dataset = encoded_dataset["train"]
val_dataset   = encoded_dataset["validation"]
test_dataset  = encoded_dataset["test"]

print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ─── 4. MODEL ARCHITECTURE (FFT L/14 + GATED FUSION + ALIGNMENT) ──────────────
class CLIPForMFND(nn.Module):
    def __init__(self, model_name, num_labels=2, lambda_weight=0.1, delta=0.5):
        super().__init__()
        # Load mô hình Large (Full Tham số)
        self.clip = CLIPModel.from_pretrained(model_name)
        embed_dim = self.clip.config.projection_dim # L/14 có embed_dim = 768

        # 1. Gated Fusion & Alignment Head
        self.cross_attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=8, batch_first=True)
        self.W_g = nn.Linear(embed_dim * 2, embed_dim)
        self.classifier = nn.Linear(embed_dim, num_labels)

        self.lambda_weight = lambda_weight
        self.delta = delta
        self.align_loss_fn = nn.CosineEmbeddingLoss(margin=delta)

    # 🟢 THÊM HÀM NÀY ĐỂ SỬA LỖI GRADIENT CHECKPOINTING
    def gradient_checkpointing_enable(self, **kwargs):
        self.clip.gradient_checkpointing_enable(**kwargs)

    def forward(self, input_ids=None, attention_mask=None, pixel_values=None, labels=None):
        outputs = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            return_dict=True
        )
        h_T, h_I = outputs.text_embeds, outputs.image_embeds

        # --- GATED MODALITY FUSION ---
        attn_out, _ = self.cross_attention(h_T.unsqueeze(1), h_I.unsqueeze(1), h_I.unsqueeze(1))
        z = attn_out.squeeze(1)

        gate = torch.sigmoid(self.W_g(torch.cat([h_T, h_I], dim=1)))
        h_f = gate * z

        # --- CLASSIFICATION & LOSS ---
        logits = self.classifier(h_f)
        loss = None
        if labels is not None:
            loss_cls = nn.CrossEntropyLoss()(logits, labels)
            # Alignment Loss chuẩn: labels * 2 - 1
            loss_align = self.align_loss_fn(h_T, h_I, labels.float() * 2 - 1)
            loss = loss_cls + (self.lambda_weight * loss_align)

        return SequenceClassifierOutput(loss=loss, logits=logits)

import gc; gc.collect(); torch.cuda.empty_cache()

model = CLIPForMFND(model_name).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Total params    : {total_params:,} (~{total_params/1e6:.2f}M)")
print(f"📊 Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")

# ─── 5. METRICS & TRAINING ────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds  = np.argmax(probs, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "auc":       roc_auc_score(labels, probs[:, 1])
    }

torch.load = functools.partial(torch.load, weights_only=False)

training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    overwrite_output_dir=True,
    num_train_epochs=5,
    # CẤU HÌNH CỨU VRAM CHO L/14
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8, # Effective Batch Size = 2 * 8 = 16
    per_device_eval_batch_size=8,
    gradient_checkpointing=True, # BẬT CHỐNG TRÀN VRAM
    learning_rate=1e-5, # FFT nên để 1e-5
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("\n🚀 Bắt đầu huấn luyện Baseline (L/14 FFT)...")
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
start_train = time.time()

trainer.train()

trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
train_time_min = (time.time() - start_train) / 60
print(f"✅ Đã lưu mô hình tại: {SAVE_DIR} | Thời gian train: {train_time_min:.2f} phút")

# ─── 6. TEST EVALUATION (IFND) & HARDWARE STATS ───────────────────────────────
model.eval()

# Tính độ trễ (Latency) & VRAM
dummy_image = Image.new("RGB", (224, 224), (128, 128, 128))
dummy_text  = "This is a sample news headline."
inputs = processor(text=dummy_text, images=dummy_image, truncation=True, padding="max_length", max_length=77, return_tensors="pt")
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)
pixel_values = inputs["pixel_values"].to(device)

with torch.no_grad():
    for _ in range(10): model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)
latencies = []
with torch.no_grad():
    for _ in range(100):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)
        if device == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

vram = torch.cuda.max_memory_allocated() / 1024**3 if device == "cuda" else 0

print(f"\n{'='*50}")
print(f"⚡ Hardware & Efficiency Stats (L/14 FFT)")
print(f"Latency median  : {np.median(latencies):.2f} ms/sample")
print(f"VRAM (peak)     : {vram:.2f} GB")
print(f"{'='*50}")

print("\n📊 Evaluating on IFND TEST split...")
results = trainer.predict(test_dataset)
logits  = results.predictions
labels  = results.label_ids
probs   = torch.softmax(torch.tensor(logits), dim=1).numpy()
preds   = np.argmax(probs, axis=1)

p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
print(f"\n[TEST RESULT]")
print(f" Accuracy : {accuracy_score(labels, preds)*100:.2f}%")
print(f" F1 Score : {f1*100:.2f}% | AUC: {roc_auc_score(labels, probs[:, 1]):.4f}")

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Save to: /content/drive/MyDrive/CLIP_ViT_L14_IFND_Final
⏳ Loading dataset IFND-multimodal...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


⏳ Preprocessing dataset...


Preprocessing:   0%|          | 0/1053 [00:00<?, ? examples/s]

...

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.557200,0.469579,0.809886,0.801193,1.000000,0.889625,0.972329
2,0.411900,0.369780,0.829848,0.818922,0.998759,0.899944,0.885664
3,0.349500,0.373850,0.830798,0.842795,0.957816,0.896632,0.886456
4,0.265500,0.494037,0.812738,0.858657,0.904467,0.880967,0.907046


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

✅ Đã lưu mô hình tại: /content/drive/MyDrive/CLIP_ViT_L14_IFND_Final | Thời gian train: 87.41 phút

⚡ Hardware & Efficiency Stats (L/14 FFT)
Latency median  : 44.77 ms/sample
VRAM (peak)     : 8.05 GB

📊 Evaluating on IFND TEST split...



[TEST RESULT]
 Accuracy : 83.00%
 F1 Score : 90.01% | AUC: 0.8743


In [ ]:
# ─── 6. TEST EVALUATION (IFND) & HARDWARE STATS & SAVE ────────────────────────
import json
model.eval()

# 1. Tính độ trễ (Latency) & VRAM
dummy_image = Image.new("RGB", (224, 224), (128, 128, 128))
dummy_text  = "This is a sample news headline for inference measurement."
inputs = processor(text=dummy_text, images=dummy_image, truncation=True, padding="max_length", max_length=77, return_tensors="pt")
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)
pixel_values = inputs["pixel_values"].to(device)

# Warmup GPU
with torch.no_grad():
    for _ in range(10): model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)

latencies = []
with torch.no_grad():
    for _ in range(100):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)
        if device == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

vram_gb = torch.cuda.max_memory_allocated() / 1024**3 if device == "cuda" else 0

print(f"\n{'='*50}")
print(f"⚡ Hardware & Efficiency Stats (L/14 FFT)")
print(f"Latency median  : {np.median(latencies):.2f} ms/sample")
print(f"VRAM (peak)     : {vram_gb:.2f} GB")
print(f"{'='*50}")

# 2. Đánh giá trên tập TEST
print("\n📊 Evaluating on IFND TEST split...")
results = trainer.predict(test_dataset)
logits  = results.predictions
labels  = results.label_ids
probs   = torch.softmax(torch.tensor(logits), dim=1).numpy()
preds   = np.argmax(probs, axis=1)

p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
acc = accuracy_score(labels, preds)
auc = roc_auc_score(labels, probs[:, 1])

# 3. Cấu trúc lại kết quả để lưu JSON
final_results = {
    "Model": "CLIP ViT-L/14",
    "Method": "Full Fine-Tuning (FFT)",
    "Dataset": "IFND-multimodal",
    "Hardware_Stats": {
        "Total_Params_M": round(total_params / 1e6, 2),
        "Trainable_Params_M": round(trainable_params / 1e6, 2),
        "Training_Time_Min": round(train_time_min, 2),
        "Peak_VRAM_GB": round(vram_gb, 2),
        "Peak_VRAM_MB": round(vram_gb * 1024, 2),
        "Latency_P50_ms": round(np.median(latencies), 2),
        "Latency_P90_ms": round(np.percentile(latencies, 90), 2)
    },
    "Test_Metrics": {
        "Accuracy": round(acc * 100, 2),
        "Precision": round(p * 100, 2),
        "Recall": round(r * 100, 2),
        "F1_Score": round(f1 * 100, 2),
        "AUC": round(auc, 4)
    }
}

# Lưu file JSON vào Drive
json_path = os.path.join(SAVE_DIR, "results_L14_FFT_IFND.json")
with open(json_path, "w") as f:
    json.dump(final_results, f, indent=4)

print(f"\n[TEST RESULT]")
print(f" Accuracy : {acc*100:.2f}%")
print(f" F1 Score : {f1*100:.2f}% | AUC: {auc:.4f}")
print(f"\n💾 Đã lưu kết quả tại: {json_path}")
print(f"{'='*50}")

# ─── 7. AUTO-DISCONNECT ──────────────────────────────────────────────────────
print("⏳ Đang ngắt kết nối phiên làm việc để giải phóng GPU...")
from google.colab import runtime
time.sleep(10) # Đợi đồng bộ Drive
runtime.unassign()


⚡ Hardware & Efficiency Stats (L/14 FFT)
Latency median  : 82.49 ms/sample
VRAM (peak)     : 8.05 GB

📊 Evaluating on IFND TEST split...



[TEST RESULT]
 Accuracy : 83.00%
 F1 Score : 90.01% | AUC: 0.8743

💾 Đã lưu kết quả tại: /content/drive/MyDrive/CLIP_ViT_L14_IFND_Final/results_L14_FFT_IFND.json
⏳ Đang ngắt kết nối phiên làm việc để giải phóng GPU...
